<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Aggregate and compare policy indicators across one policy-assessment portfolio run.
- **Primary inputs**: `runs/assessment_policies/<assessment_run>/<policy>/policies/indicator.csv`.
- **Primary outputs**: `indicator_all_policies.csv`, `policy_summary_ap.csv`, `policy_summary_zp.csv`, and portfolio plots.
- **Refactor role**: Replace exploratory ad-hoc cells with a clear, reusable workflow and explicit markdown guidance for sharing.
- **Data policy**: This notebook reads run artifacts in place and writes summary files into the selected run folder.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Resolve one `assessment_policies` run and load all policy `indicator.csv` files.
2. Build and export a consolidated indicator table across policies.
3. Generate line plots for cost-effectiveness and margin indicators.
4. Generate cost-benefit stacked bars from NPV rows.
5. Export AP and ZP summary tables for communication.

### Inputs
- `project/analysis/post_processing/policy_assessment/runs/assessment_policies/<assessment_run>/<policy>/policies/indicator.csv`

### Outputs
- `project/analysis/post_processing/policy_assessment/runs/assessment_policies/<assessment_run>/indicator_all_policies.csv`
- `project/analysis/post_processing/policy_assessment/runs/assessment_policies/<assessment_run>/policy_summary_ap.csv`
- `project/analysis/post_processing/policy_assessment/runs/assessment_policies/<assessment_run>/policy_summary_zp.csv`
- `project/analysis/post_processing/policy_assessment/runs/assessment_policies/<assessment_run>/*.png`

### How To Reuse
- Update `run_name` in the configuration cell.
- Optionally adapt variable lists and `policy_group_map` for your portfolio definition.


# Summarize policy portfolio indicators

Use this notebook as the shareable reporting workflow for one policy-assessment run.
Run all cells top-to-bottom after selecting `run_name` below.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import pandas as pd

from project.analysis.post_processing.shared.io_indicators import load_indicators_from_children
from project.analysis.post_processing.shared.paths import resolve_run
from project.input.resources import resources_data
from project.utils import make_stackedbar_plot


## Configuration

Set the run and optional plotting/export controls.


In [ ]:
# Run selection
# Supported values:
# - "assessment_YYYYMMDD_HHMMSS"
# - "assessment_policies/assessment_YYYYMMDD_HHMMSS"
run_name = "assessment_20240117_093354"
save_figures = True

# Cross-policy indicator panel
indicator_variables = [
    "Cost effectiveness (euro/kWh)",
    "Cost effectiveness standard (euro/kWh)",
    "Cost effectiveness carbon (euro/tCO2)",
    "Leverage (%)",
    "Intensive margin insulation (%)",
    "Intensive margin insulation energy (%)",
    "Extensive margin insulation (%)",
]

# Optional grouping to simplify portfolio communication on charts
policy_group_map = {
    "mpr_multifamily": "Multi-family",
    "mpr_multifamily_updated-mpr_multifamily_deep": "Multi-family",
    "cite": "Single-family",
    "mpr": "Single-family",
    "mpr_efficacite": "Single-family",
    "mpr_serenite": "Deep renovation",
    "mpr_performance": "Deep renovation",
    "cee": "White certificate",
    "reduced_vat": "Reduced VAT",
    "zero_interest_loan": "Zero-interest loan",
}

indicator_pairs = [
    ["Cost effectiveness (euro/kWh)", "Cost effectiveness standard (euro/kWh)"],
    ["Extensive margin insulation (%)", "Intensive margin insulation energy (%)"],
]

drop_policy_columns = ["carbon_tax"]

# NPV/cost-benefit extraction
npv_variables = [
    "Investment",
    "Embodied emission",
    "Opportunity cost",
    "Energy saving",
    "Comfort EE",
    "Comfort prices",
    "Emission saving",
    "Health cost",
]
npv_plot_titles = {
    "AP-1": "Cost-benefits analysis with policy interactions (Billion euro)",
    "ZP+1": "Cost-benefits analysis without policy interaction (Billion euro)",
}
npv_policy_order = ["Subsidies", "White certificate", "Zero interest", "Carbon tax", "Mandatory"]

# AP/ZP summary export
summary_variables = [
    "Consumption saving (TWh)",
    "Consumption saving (%)",
    "Consumption standard saving (TWh)",
    "Consumption standard saving (kWh/m2)",
    "Consumption standard saving (%)",
    "Emission saving (MtCO2)",
    "Emission saving (%)",
    "Cumulated emission saving (MtCO2)",
    "Energy poverty diff (Million)",
    "Energy poverty diff (%)",
    "Investment heater diff (Billion euro)",
    "Investment heater diff (%)",
    "Subsidies heater diff (Billion euro)",
    "Subsidies heater diff (%)",
    "Investment insulation diff (Billion euro)",
    "Investment insulation diff (%)",
    "Subsidies insulation diff (Billion euro)",
    "Subsidies insulation diff (%)",
]


## Helpers

Reusable functions for loading, reshaping, plotting, and exporting tables.


In [ ]:
def _candidate_assessment_roots():
    return [
        resolve_run("policy_assessment", "assessment_policies"),
        Path("runs") / "assessment_policies",
    ]


def list_available_assessment_runs():
    runs = set()
    for root in _candidate_assessment_roots():
        if not root.exists():
            continue
        runs.update([child.name for child in root.iterdir() if child.is_dir()])
    return sorted(runs)


def resolve_assessment_run(run_name):
    run_name = str(run_name).strip()
    run_path = Path(run_name)

    candidates = [
        resolve_run("policy_assessment", run_name),
        Path("runs") / run_name,
    ]

    if run_path.parts and run_path.parts[0] == "assessment_policies":
        candidates.append(Path("runs") / run_path)
    else:
        candidates.extend([
            resolve_run("policy_assessment", str(Path("assessment_policies") / run_name)),
            Path("runs") / "assessment_policies" / run_name,
        ])

    unique_candidates = []
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        unique_candidates.append(candidate)
        seen.add(key)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate.resolve()

    available = list_available_assessment_runs()
    checked = "\n".join([" - {}".format(path) for path in unique_candidates])
    raise FileNotFoundError(
        "Run folder not found for run_name='{}'.\nChecked locations:\n{}\nAvailable runs: {}".format(
            run_name,
            checked,
            available,
        )
    )


def to_display_name(name):
    return name.replace("_", " ").replace("-", " ").title()


def build_indicator_panel(indicator_bundle, variables):
    frames = {}

    for indicator in variables:
        by_policy = {}
        for policy_name, table in indicator_bundle.items():
            if indicator not in table.index:
                continue
            series = table.loc[indicator].dropna()
            if not series.empty:
                by_policy[policy_name] = series

        if by_policy:
            frames[indicator] = pd.DataFrame(by_policy)

    if not frames:
        raise ValueError("No configured indicators were found in loaded policy tables.")

    panel = pd.concat(frames, axis=0)
    panel.index.names = ["Indicator", "Year"]
    panel = panel.sort_index(axis=1)
    return panel


def prepare_indicator_plot_table(panel, remove_policies=None):
    plot_table = panel.reset_index().copy()
    year_str = plot_table["Year"].astype(str)

    plot_table["Year_Number"] = year_str.str.extract(r"(\d+)", expand=False)
    plot_table["Year_Number"] = pd.to_numeric(plot_table["Year_Number"], errors="coerce").fillna(0).astype(int)
    plot_table["AP_ZP"] = year_str.str.extract(r"(AP|ZP)", expand=False)

    if remove_policies:
        for policy in remove_policies:
            if policy in plot_table.columns:
                plot_table = plot_table.drop(columns=[policy])

    return plot_table


def group_policy_columns(df, group_map):
    value_cols = [col for col in df.columns if col not in ["Year_Number", "AP_ZP"]]
    if not group_map:
        return df, value_cols

    renamed = {col: group_map.get(col, col) for col in value_cols}
    grouped_values = df[value_cols].T.groupby(renamed, sort=False).first().T
    grouped_df = pd.concat([df[["Year_Number", "AP_ZP"]], grouped_values], axis=1)
    return grouped_df, list(grouped_values.columns)


def format_unit_axis(ax, values, unit):
    if unit == "%":
        ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: "{:.0%}".format(y)))
        return

    max_value = pd.Series(values).dropna().max()
    if pd.isna(max_value):
        max_value = 0

    if max_value > 10:
        template = "{:.0f} {}"
    elif max_value > 0.6:
        template = "{:.1f} {}"
    else:
        template = "{:.2f} {}"

    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: template.format(y, unit)))


def indicator_slug(indicator):
    base = indicator.split(" (")[0].strip().lower()
    return base.replace(" ", "_").replace("/", "_")


def make_indicator_line_chart(plot_table, indicators, group_map=None, output_folder=None):
    if isinstance(indicators, str):
        indicators = [indicators]

    fig, axes = plt.subplots(1, len(indicators), figsize=(8 * len(indicators), 6), sharey=(len(indicators) > 1))
    if len(indicators) == 1:
        axes = [axes]

    policy_handles = {}

    for ax, indicator in zip(axes, indicators):
        df = plot_table[plot_table["Indicator"] == indicator].copy()
        if df.empty:
            raise ValueError("Indicator '{}' was not found in plot table.".format(indicator))

        df = df[df["Year_Number"] != 1]
        df = df.drop(columns=["Indicator", "Year"])
        df, value_cols = group_policy_columns(df, group_map)

        ap = df[df["AP_ZP"] == "AP"].sort_values("Year_Number")
        zp = df[df["AP_ZP"] == "ZP"].sort_values("Year_Number")

        for idx, col in enumerate(value_cols):
            color = plt.cm.tab10(idx % 10)
            label = col.replace("_", " ").title()
            policy_handles[col] = Line2D([0], [0], color=color, linestyle="-", marker="o", label=label)

            if not ap.empty:
                ax.plot(ap["Year_Number"], ap[col], linestyle="-", marker="o", color=color)
            if not zp.empty:
                ax.plot(zp["Year_Number"], zp[col], linestyle=":", marker="x", color=color)

        unit = indicator.split("(")[-1].replace(")", "").replace("euro", "€") if "(" in indicator else ""
        format_unit_axis(ax, df[value_cols].to_numpy().ravel(), unit)

        ax.set_title(indicator.split(" (")[0])
        ax.set_xlabel("Year index")
        ax.set_ylim(bottom=0)
        ax.tick_params(which="both", bottom=False, top=False, labelsize=12)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(False)

    style_handles = [
        Line2D([0], [0], color="black", linestyle="-", marker="o", label="With interaction (AP)"),
        Line2D([0], [0], color="black", linestyle=":", marker="x", label="Without interaction (ZP)"),
    ]

    if policy_handles:
        fig.legend(
            handles=list(policy_handles.values()),
            title="Policies",
            loc="center left",
            bbox_to_anchor=(0.99, 0.72),
            frameon=False,
        )

    fig.legend(
        handles=style_handles,
        title="Method",
        loc="center left",
        bbox_to_anchor=(0.99, 0.38),
        frameon=False,
    )

    fig.tight_layout(rect=[0, 0, 0.86, 1])

    save_path = None
    if output_folder is not None:
        name = "_".join([indicator_slug(i) for i in indicators])
        save_path = output_folder / "{}.png".format(name)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()

    return save_path


def build_npv_table(indicator_bundle, variables, scenario_key, policy_order=None):
    result = {}

    for policy_name, table in indicator_bundle.items():
        if scenario_key not in table.columns:
            continue
        series = table.reindex(variables)[scenario_key].dropna()
        if not series.empty:
            result[to_display_name(policy_name)] = series

    if not result:
        raise ValueError("No NPV rows found for scenario key '{}'".format(scenario_key))

    table = -pd.DataFrame(result)

    if policy_order:
        existing = [name for name in policy_order if name in table.columns]
        others = [name for name in table.columns if name not in existing]
        table = table[existing + others]

    return table


def build_policy_summary_table(indicator_bundle, variables, scenario_key):
    result = {}

    for policy_name, table in indicator_bundle.items():
        if scenario_key not in table.columns:
            continue
        result[to_display_name(policy_name)] = table.reindex(variables)[scenario_key]

    if not result:
        raise ValueError("No policy summary rows found for scenario key '{}'".format(scenario_key))

    summary = pd.DataFrame(result)

    if {
        "Consumption saving (TWh)",
        "Consumption standard saving (TWh)",
    }.issubset(summary.index):
        summary.loc["Performance gap", :] = (
            summary.loc["Consumption saving (TWh)", :]
            / summary.loc["Consumption standard saving (TWh)", :]
        )

    return summary


## 1) Load run data

Resolve the selected run and load all policy indicator tables.


In [ ]:
run_folder = resolve_assessment_run(run_name)
indicator_bundle = load_indicators_from_children(run_folder)

if not indicator_bundle:
    raise FileNotFoundError("No policies/indicator.csv files found in {}".format(run_folder))

print("Run folder:", run_folder)
print("Policies loaded:", len(indicator_bundle))
print("Policy list:", sorted(indicator_bundle.keys()))


## 2) Build consolidated indicator panel

Create and export `indicator_all_policies.csv`, then prepare a plotting table with parsed AP/ZP metadata.


In [ ]:
indicator_panel = build_indicator_panel(indicator_bundle, indicator_variables)
indicator_plot_table = prepare_indicator_plot_table(indicator_panel, remove_policies=drop_policy_columns)

indicator_all_path = run_folder / "indicator_all_policies.csv"
indicator_panel.to_csv(indicator_all_path)

print("Saved:", indicator_all_path)
indicator_plot_table.head()


## 3) Create indicator line charts

- Solid lines: AP (with policy interaction)
- Dotted lines: ZP (without interaction)


In [ ]:
saved_charts = []

for indicator in indicator_variables:
    save_path = make_indicator_line_chart(
        indicator_plot_table,
        indicator,
        group_map=policy_group_map,
        output_folder=run_folder if save_figures else None,
    )
    if save_path is not None:
        saved_charts.append(save_path)

for pair in indicator_pairs:
    save_path = make_indicator_line_chart(
        indicator_plot_table,
        pair,
        group_map=policy_group_map,
        output_folder=run_folder if save_figures else None,
    )
    if save_path is not None:
        saved_charts.append(save_path)

if saved_charts:
    print("Saved line charts:")
    for path in saved_charts:
        print(" -", path)


## 4) Create NPV/cost-benefit stacked bars

Build AP and ZP cost-benefit tables and export figure files.


In [ ]:
npv_tables = {}

for scenario_key, title in npv_plot_titles.items():
    npv_table = build_npv_table(
        indicator_bundle,
        variables=npv_variables,
        scenario_key=scenario_key,
        policy_order=npv_policy_order,
    )
    npv_tables[scenario_key] = npv_table

    save_path = run_folder / "cost_benefits_analysis_{}.png".format(scenario_key) if save_figures else None

    make_stackedbar_plot(
        npv_table.T,
        title,
        ncol=3,
        ymin=None,
        format_y=lambda y, _: "{:.0f} B€".format(y),
        hline=0,
        colors=resources_data["colors"],
        scatterplot=npv_table.sum(),
        save=str(save_path) if save_path is not None else None,
        rotation=0,
        left=1.3,
        fontxtick=15,
    )

    if save_path is not None:
        print("Saved:", save_path)

npv_tables["AP-1"]


## 5) Export AP/ZP policy summary tables

Generate final communication tables for AP and ZP variants.


In [ ]:
policy_summary_ap = build_policy_summary_table(indicator_bundle, summary_variables, scenario_key="AP-1")
policy_summary_zp = build_policy_summary_table(indicator_bundle, summary_variables, scenario_key="ZP+1")

ap_path = run_folder / "policy_summary_ap.csv"
zp_path = run_folder / "policy_summary_zp.csv"

policy_summary_ap.to_csv(ap_path)
policy_summary_zp.to_csv(zp_path)

print("Saved:", ap_path)
print("Saved:", zp_path)

policy_summary_ap


## Deliverables checklist

After running the notebook, verify these files in the run folder:
- `indicator_all_policies.csv`
- `policy_summary_ap.csv`
- `policy_summary_zp.csv`
- `cost_benefits_analysis_AP-1.png`
- `cost_benefits_analysis_ZP+1.png`
- Indicator line charts (`*.png`)
